# ECON 1307 — Clase 8
## Ciclo de vida de los datos: Carga, Limpieza, Construcción de Variables e Indexación

**Objetivo:** Trabajar con datos reales de COVID-19 en Colombia para practicar un flujo completo de:
1. **Carga** desde una fuente externa (GitHub)
2. **Inspección** profunda del dataset
3. **Limpieza** (columnas, tipos, duplicados, valores faltantes, inconsistencias de texto)
4. **Construcción de variables** derivadas útiles para análisis
5. **Indexación y filtrado** para responder preguntas de interés

⚠️ Regla de oro: **Antes de limpiar, siempre hay que inspeccionar.** No se puede arreglar lo que no se entiende.

---

In [1]:
import pandas as pd
import numpy as np

print("Librerías cargadas")

Librerías cargadas


---
## 1. Carga de datos

Vamos a cargar un dataset real de casos de COVID-19 en Colombia, alojado en GitHub.  
Son ~124,000 registros de casos individuales con información demográfica, geográfica y temporal.

`pd.read_csv()` puede recibir una URL directamente. El parámetro `low_memory=False` evita advertencias cuando las columnas tienen tipos mixtos.

In [2]:
url = "https://github.com/Ambg0231/Uniandes/raw/refs/heads/main/COVID-19.csv"
df = pd.read_csv(url, encoding="utf-8", low_memory=False)

print("Dimensiones del dataset:", df.shape)
print(f"  → {df.shape[0]:,} filas (casos)")
print(f"  → {df.shape[1]} columnas (variables)")

Dimensiones del dataset: (123782, 23)
  → 123,782 filas (casos)
  → 23 columnas (variables)


---
## 2. Inspección inicial

Antes de tocar los datos, necesitamos entender:
- ¿Qué columnas hay y de qué tipo son?
- ¿Cuántos valores faltantes hay?
- ¿Cómo lucen las primeras filas?
- ¿Qué valores toman las variables categóricas?
- ¿Hay inconsistencias?

In [3]:
# 2.1 Primeras filas
df.head()

,fecha reporte web,ID de caso,Fecha de notificación,Código DIVIPOLA departamento,Nombre departamento,Código DIVIPOLA municipio,Nombre municipio,Edad,Unidad de medida de edad,Sexo,...,Código ISO del país,Nombre del país,Recuperado,Fecha de inicio de síntomas,Fecha de muerte,Fecha de diagnóstico,Fecha de recuperación,Tipo de recuperación,Pertenencia étnica,Nombre del grupo étnico
0,2020-12-24 00:00:00,1556979,2020-12-22 00:00:00,76,VALLE,76001,CALI,67,1,F,...,NaN,NaN,Recuperado,2020-12-21 00:00:00,NaN,2020-12-23 00:00:00,2021-01-04 00:00:00,Tiempo,6,NaN
1,2020-12-24 00:00:00,1556980,2020-12-19 00:00:00,76,VALLE,76001,CALI,66,1,F,...,NaN,NaN,Recuperado,2020-12-07 00:00:00,NaN,2020-12-23 00:00:00,2020-12-25 00:00:00,Tiempo,6,NaN
2,2020-12-24 00:00:00,1556981,2020-12-19 00:00:00,76,VALLE,76001,CALI,68,1,F,...,NaN,NaN,Recuperado,2020-12-18 00:00:00,NaN,2020-12-22 00:00:00,2021-01-01 00:00:00,Tiempo,6,NaN
3,2020-12-24 00:00:00,1556982,2020-12-22 00:00:00,76,VALLE,76001,CALI,74,1,F,...,NaN,NaN,Fallecido,2020-12-17 00:00:00,2020-12-30 00:00:00,2020-12-23 00:00:00,NaN,NaN,6,NaN
4,2020-12-24 00:00:00,1556983,2020-12-22 00:00:00,76,VALLE,76001,CALI,65,1,F,...,NaN,NaN,Recuperado,2020-12-21 00:00:00,NaN,2020-12-23 00:00:00,2021-01-04 00:00:00,Tiempo,6,NaN


In [4]:
# 2.2 Nombres de todas las columnas
print("Columnas del dataset:")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}. {col}")

Columnas del dataset:
   0. fecha reporte web
   1. ID de caso
   2. Fecha de notificación
   3. Código DIVIPOLA departamento
   4. Nombre departamento
   5. Código DIVIPOLA municipio
   6. Nombre municipio
   7. Edad
   8. Unidad de medida de edad
   9. Sexo
  10. Tipo de contagio
  11. Ubicación del caso
  12. Estado
  13. Código ISO del país
  14. Nombre del país
  15. Recuperado
  16. Fecha de inicio de síntomas
  17. Fecha de muerte
  18. Fecha de diagnóstico
  19. Fecha de recuperación
  20. Tipo de recuperación
  21. Pertenencia étnica
  22. Nombre del grupo étnico


In [5]:
# 2.3 Tipos de datos y conteo de no-nulos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123782 entries, 0 to 123781
Data columns (total 23 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   fecha reporte web             123782 non-null  object 
 1   ID de caso                    123782 non-null  int64  
 2   Fecha de notificación         123782 non-null  object 
 3   Código DIVIPOLA departamento  123782 non-null  int64  
 4   Nombre departamento           123782 non-null  object 
 5   Código DIVIPOLA municipio     123782 non-null  int64  
 6   Nombre municipio              123782 non-null  object 
 7   Edad                          123782 non-null  int64  
 8   Unidad de medida de edad      123782 non-null  int64  
 9   Sexo                          123782 non-null  object 
 10  Tipo de contagio              123782 non-null  object 
 11  Ubicación del caso            123078 non-null  object 
 12  Estado                        123078 non-nul

In [6]:
# 2.4 Valores faltantes: conteo y porcentaje
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": pct_nulos
}).sort_values("porcentaje", ascending=False)

print("Valores faltantes por columna:")
resumen_nulos

Valores faltantes por columna:


,nulos,porcentaje
Código ISO del país,123695,99.93
Nombre del país,123695,99.93
Nombre del grupo étnico,121561,98.21
Fecha de muerte,119583,96.61
Fecha de inicio de síntomas,9202,7.43
Fecha de recuperación,4051,3.27
Tipo de recuperación,4051,3.27
Estado,704,0.57
Ubicación del caso,704,0.57
Recuperado,556,0.45


In [7]:
# 2.5 Explorar variables categóricas clave
# Veamos qué valores toma cada variable categórica importante

print("=" * 50)
print("Valores únicos en 'Sexo':")
print(df["Sexo"].value_counts(dropna=False))

print("\n" + "=" * 50)
print("Valores únicos en 'Tipo de contagio':")
print(df["Tipo de contagio"].value_counts(dropna=False))

Valores únicos en 'Sexo':
Sexo
F    62430
M    61352
Name: count, dtype: int64

Valores únicos en 'Tipo de contagio':
Tipo de contagio
Comunitaria    94594
Relacionado    29101
Importado         87
Name: count, dtype: int64


In [8]:
# 2.6 Detección de inconsistencias en texto
# Miremos con atención estas columnas: ¿ven algo raro?

print("Valores en 'Recuperado':")
print(df["Recuperado"].value_counts(dropna=False))

print("\nValores en 'Estado':")
print(df["Estado"].value_counts(dropna=False))

print("\nValores en 'Ubicación del caso':")
print(df["Ubicación del caso"].value_counts(dropna=False))

Valores en 'Recuperado':
Recuperado
Recuperado    119731
Fallecido       3483
NaN              556
fallecido         12
Name: count, dtype: int64

Valores en 'Estado':
Estado
Leve         119576
Fallecido      3495
NaN             704
leve              7
Name: count, dtype: int64

Valores en 'Ubicación del caso':
Ubicación del caso
Casa         119576
Fallecido      3495
NaN             704
casa              7
Name: count, dtype: int64


**¡Problema detectado!** Noten que hay inconsistencias en la capitalización:
- `Recuperado`: aparece "Fallecido" (3,483 veces) **y** "fallecido" (12 veces) → son la misma categoría pero pandas las trata como distintas
- `Estado`: aparece "Leve" (119,576) **y** "leve" (7) → mismo problema
- `Ubicación del caso`: aparece "Casa" (119,576) **y** "casa" (7)

Esto es un error muy común en datos reales. Si no lo corregimos, nuestros conteos y agrupaciones estarán mal.

In [9]:
# 2.7 Inspeccionar la variable 'Unidad de medida de edad'
# Esta columna es clave: ¿la edad está siempre en años?

print("Valores en 'Unidad de medida de edad':")
print(df["Unidad de medida de edad"].value_counts())
print("\n1 = Años, 2 = Meses, 3 = Días")

Valores en 'Unidad de medida de edad':
Unidad de medida de edad
1    123347
2       387
3        48
Name: count, dtype: int64

1 = Años, 2 = Meses, 3 = Días


In [10]:
# Veamos los casos donde la edad NO está en años
print("Casos con edad en MESES (unidad=2):")
print(df[df["Unidad de medida de edad"] == 2][["Edad", "Unidad de medida de edad"]].head(10))

print("\nCasos con edad en DÍAS (unidad=3):")
print(df[df["Unidad de medida de edad"] == 3][["Edad", "Unidad de medida de edad"]].head(10))

Casos con edad en MESES (unidad=2):
      Edad  Unidad de medida de edad
884      6                         2
1476     3                         2
2126     6                         2
2252     2                         2
2261     1                         2
2450     4                         2
2493     2                         2
3168     1                         2
3170     1                         2
3220     5                         2

Casos con edad en DÍAS (unidad=3):
       Edad  Unidad de medida de edad
2318     17                         3
2956      5                         3
8495     26                         3
10375     1                         3
16131    29                         3
16755    17                         3
20001     1                         3
26393     1                         3
28088    14                         3
31652    19                         3


**¡Otro problema!** Hay 387 casos donde la edad está en **meses** y 48 donde está en **días**.  
Si no convertimos todo a la misma unidad, un bebé de 6 meses aparecería como si tuviera 6 años.

In [11]:
# 2.8 Estadísticas descriptivas de la edad
print("Estadísticas de la columna 'Edad' (sin corregir unidad):")
print(df["Edad"].describe())

print("\nValor máximo de edad:", df["Edad"].max())
print("Valor mínimo de edad:", df["Edad"].min())

Estadísticas de la columna 'Edad' (sin corregir unidad):
count    123782.000000
mean         40.003353
std          18.045711
min           1.000000
25%          27.000000
50%          37.000000
75%          52.000000
max         108.000000
Name: Edad, dtype: float64

Valor máximo de edad: 108
Valor mínimo de edad: 1


In [12]:
# 2.9 Inspeccionar 'Pertenencia étnica'
# Está codificada como número. Según el diccionario de datos del INS:
# 1=Indígena, 2=ROM/Gitano, 3=Raizal, 4=Palenquero, 5=Negro/Afrocolombiano, 6=Otro

print("Valores en 'Pertenencia étnica':")
print(df["Pertenencia étnica"].value_counts().sort_index())

Valores en 'Pertenencia étnica':
Pertenencia étnica
1      2234
2         1
3        31
5      4954
6    116562
Name: count, dtype: int64


### Resumen de la inspección

| Hallazgo | Problema | Acción necesaria |
|---|---|---|
| Nombres de columnas | Tildes, espacios, mayúsculas | Normalizar |
| Fechas | Son texto (`object`) | Convertir a `datetime` |
| `Código ISO del país`, `Nombre del país` | >99% nulos | Eliminar |
| `Recuperado`, `Estado`, `Ubicación` | Inconsistencias de capitalización ("Fallecido" vs "fallecido") | Normalizar texto |
| `Unidad de medida de edad` | 387 edades en meses, 48 en días | Convertir todo a años |
| `Pertenencia étnica` | Codificada como número | Mapear a nombres descriptivos |
| Duplicados | Hay que verificar | Revisar por ID de caso |

---

## 3. Limpieza paso a paso

Vamos a hacer una copia para trabajar, así siempre tenemos los datos originales intactos.

In [13]:
# Copia de trabajo
df_clean = df.copy()
print("Copia creada. Dimensiones:", df_clean.shape)

Copia creada. Dimensiones: (123782, 23)


### 3.1 Normalizar nombres de columnas

Los nombres con espacios, tildes y mayúsculas generan errores al escribir código. Vamos a estandarizarlos.

In [14]:
df_clean.columns = (
    df_clean.columns
    .str.strip()             # Quitar espacios al inicio/final
    .str.lower()             # Todo a minúsculas
    .str.replace(" ", "_")   # Espacios por guión bajo
    .str.replace("á", "a")   # Quitar tildes
    .str.replace("é", "e")
    .str.replace("í", "i")
    .str.replace("ó", "o")
    .str.replace("ú", "u")
)

print("Columnas normalizadas:")
for i, col in enumerate(df_clean.columns):
    print(f"  {i:2d}. {col}")

Columnas normalizadas:
   0. fecha_reporte_web
   1. id_de_caso
   2. fecha_de_notificacion
   3. codigo_divipola_departamento
   4. nombre_departamento
   5. codigo_divipola_municipio
   6. nombre_municipio
   7. edad
   8. unidad_de_medida_de_edad
   9. sexo
  10. tipo_de_contagio
  11. ubicacion_del_caso
  12. estado
  13. codigo_iso_del_pais
  14. nombre_del_pais
  15. recuperado
  16. fecha_de_inicio_de_sintomas
  17. fecha_de_muerte
  18. fecha_de_diagnostico
  19. fecha_de_recuperacion
  20. tipo_de_recuperacion
  21. pertenencia_etnica
  22. nombre_del_grupo_etnico


### 3.2 Eliminar columnas que no aportan

Las columnas `codigo_iso_del_pais` y `nombre_del_pais` tienen >99% de nulos. No aportan información útil.

In [15]:
cols_a_eliminar = ["codigo_iso_del_pais", "nombre_del_pais"]

df_clean = df_clean.drop(columns=cols_a_eliminar)
print(f"Se eliminaron {len(cols_a_eliminar)} columnas.")
print(f"Dimensiones actuales: {df_clean.shape}")

Se eliminaron 2 columnas.
Dimensiones actuales: (123782, 21)


### 3.3 Normalizar texto en variables categóricas

Vimos que hay inconsistencias como "Fallecido" vs "fallecido". Estandaricemos todo a minúsculas.

In [16]:
# ANTES de normalizar:
print("ANTES — Valores en 'recuperado':")
print(df_clean["recuperado"].value_counts(dropna=False))

ANTES — Valores en 'recuperado':
recuperado
Recuperado    119731
Fallecido       3483
NaN              556
fallecido         12
Name: count, dtype: int64


In [17]:
# Normalizar todas las columnas de texto
cols_texto = df_clean.select_dtypes(include="object").columns

for col in cols_texto:
    df_clean[col] = df_clean[col].str.strip().str.lower()

# DESPUÉS de normalizar:
print("DESPUÉS — Valores en 'recuperado':")
print(df_clean["recuperado"].value_counts(dropna=False))

print("\nDESPUÉS — Valores en 'ubicacion_del_caso':")
print(df_clean["ubicacion_del_caso"].value_counts(dropna=False))

DESPUÉS — Valores en 'recuperado':
recuperado
recuperado    119731
fallecido       3495
NaN              556
Name: count, dtype: int64

DESPUÉS — Valores en 'ubicacion_del_caso':
ubicacion_del_caso
casa         119583
fallecido      3495
NaN             704
Name: count, dtype: int64


Ahora "Fallecido" y "fallecido" se unificaron en una sola categoría. Lo mismo con "Casa" y "casa".

### 3.4 Convertir fechas

Las fechas están como texto (`object`). Necesitamos convertirlas a `datetime` para poder calcular diferencias de tiempo.

In [18]:
cols_fecha = [
    "fecha_reporte_web",
    "fecha_de_notificacion",
    "fecha_de_inicio_de_sintomas",
    "fecha_de_muerte",
    "fecha_de_diagnostico",
    "fecha_de_recuperacion"
]

for col in cols_fecha:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

# Verificar la conversión
print("Tipos de las columnas de fecha:")
for col in cols_fecha:
    if col in df_clean.columns:
        print(f"  {col}: {df_clean[col].dtype}")

print(f"\nRango de fechas de reporte: {df_clean['fecha_reporte_web'].min()} a {df_clean['fecha_reporte_web'].max()}")

Tipos de las columnas de fecha:
  fecha_reporte_web: datetime64[ns]
  fecha_de_notificacion: datetime64[ns]
  fecha_de_inicio_de_sintomas: datetime64[ns]
  fecha_de_muerte: datetime64[ns]
  fecha_de_diagnostico: datetime64[ns]
  fecha_de_recuperacion: datetime64[ns]

Rango de fechas de reporte: 2020-05-13 00:00:00 a 2022-01-25 00:00:00


### 3.5 Corregir la edad según la unidad de medida

Descubrimos que 387 edades están en meses y 48 en días.  
Vamos a crear una columna `edad_anios` que convierta todo a años.

In [19]:
# Crear edad corregida en años
# unidad_de_medida_de_edad: 1=años, 2=meses, 3=días

df_clean["edad_anios"] = df_clean["edad"].astype(float)  # Copiar como float para permitir decimales

# Convertir meses a años
mask_meses = df_clean["unidad_de_medida_de_edad"] == 2
df_clean.loc[mask_meses, "edad_anios"] = (df_clean.loc[mask_meses, "edad"] / 12).round(1)

# Convertir días a años
mask_dias = df_clean["unidad_de_medida_de_edad"] == 3
df_clean.loc[mask_dias, "edad_anios"] = (df_clean.loc[mask_dias, "edad"] / 365).round(2)

# Verificar: ver algunos casos convertidos
print("Casos donde la edad estaba en MESES (ahora convertidos a años):")
print(df_clean.loc[mask_meses, ["edad", "unidad_de_medida_de_edad", "edad_anios"]].head(10))

print(f"\nCasos convertidos: {mask_meses.sum()} de meses, {mask_dias.sum()} de días")

Casos donde la edad estaba en MESES (ahora convertidos a años):
      edad  unidad_de_medida_de_edad  edad_anios
884      6                         2         0.5
1476     3                         2         0.2
2126     6                         2         0.5
2252     2                         2         0.2
2261     1                         2         0.1
2450     4                         2         0.3
2493     2                         2         0.2
3168     1                         2         0.1
3170     1                         2         0.1
3220     5                         2         0.4

Casos convertidos: 387 de meses, 48 de días


### 3.6 Verificar duplicados

In [20]:
# ¿Hay duplicados por ID de caso? (debería ser único)
n_dup_id = df_clean.duplicated(subset=["id_de_caso"]).sum()
print(f"Duplicados por ID de caso: {n_dup_id}")

# ¿Hay duplicados por fila completa?
n_dup_total = df_clean.duplicated().sum()
print(f"Duplicados por fila completa: {n_dup_total}")

print(f"\nIDs únicos: {df_clean['id_de_caso'].nunique():,} de {len(df_clean):,} filas")

Duplicados por ID de caso: 0
Duplicados por fila completa: 0

IDs únicos: 123,782 de 123,782 filas


En este caso no hay duplicados (¡los datos ya estaban limpios en ese aspecto!).  
Pero **siempre hay que verificar**, porque en muchos datasets reales sí los hay (como vimos en la Clase 7).

### 3.7 Manejo de valores faltantes

No todos los nulos se tratan igual. Hay que pensar **qué significa** cada nulo:

| Columna | ¿Por qué tiene nulos? | ¿Qué hacer? |
|---|---|---|
| `fecha_de_muerte` | Solo aplica a fallecidos (~119,500 nulos) | **Dejar como NaT** (es información válida: no murió) |
| `fecha_de_inicio_de_sintomas` | Puede ser asintomático (~9,200 nulos) | **Dejar como NaT** |
| `fecha_de_recuperacion` | Puede que no se haya recuperado (~4,000 nulos) | **Dejar como NaT** |
| `ubicacion_del_caso` | Dato no registrado (704 nulos) | Imputar con "desconocido" |
| `estado` | Dato no registrado (704 nulos) | Imputar con "desconocido" |
| `recuperado` | Dato no registrado (556 nulos) | Imputar con "desconocido" |
| `nombre_del_grupo_etnico` | La mayoría no tiene etnia registrada (~121,500 nulos) | Imputar con "no registrado" |

In [21]:
# Imputar categóricas con valores descriptivos
cols_imputar = {
    "ubicacion_del_caso": "desconocido",
    "estado": "desconocido",
    "recuperado": "desconocido",
    "nombre_del_grupo_etnico": "no registrado",
    "tipo_de_recuperacion": "no aplica"
}

for col, valor in cols_imputar.items():
    if col in df_clean.columns:
        n_antes = df_clean[col].isnull().sum()
        df_clean[col] = df_clean[col].fillna(valor)
        print(f"  {col}: {n_antes} nulos → imputados con '{valor}'")

# Verificar nulos restantes
print("\nNulos restantes (solo columnas con nulos):")
nulos_post = df_clean.isnull().sum()
print(nulos_post[nulos_post > 0])

  ubicacion_del_caso: 704 nulos → imputados con 'desconocido'
  estado: 704 nulos → imputados con 'desconocido'
  recuperado: 556 nulos → imputados con 'desconocido'
  nombre_del_grupo_etnico: 121561 nulos → imputados con 'no registrado'
  tipo_de_recuperacion: 4051 nulos → imputados con 'no aplica'

Nulos restantes (solo columnas con nulos):
fecha_de_inicio_de_sintomas      9202
fecha_de_muerte                119583
fecha_de_diagnostico              135
fecha_de_recuperacion            4051
dtype: int64


### 3.8 Mapear pertenencia étnica

La columna `pertenencia_etnica` tiene códigos numéricos. Vamos a mapearlos a nombres descriptivos.

In [22]:
mapa_etnia = {
    1: "Indigena",
    2: "ROM/Gitano",
    3: "Raizal",
    4: "Palenquero",
    5: "Negro/Afrocolombiano",
    6: "Otro"
}

df_clean["etnia"] = df_clean["pertenencia_etnica"].map(mapa_etnia)

print("Distribución por etnia:")
print(df_clean["etnia"].value_counts())

Distribución por etnia:
etnia
Otro                    116562
Negro/Afrocolombiano      4954
Indigena                  2234
Raizal                      31
ROM/Gitano                   1
Name: count, dtype: int64


### Resumen de la limpieza

| Paso | Qué hicimos | Herramienta |
|---|---|---|
| 3.1 | Normalizar nombres de columnas | `.str.lower()`, `.str.replace()` |
| 3.2 | Eliminar columnas con >99% nulos | `.drop(columns=)` |
| 3.3 | Normalizar texto categórico ("Fallecido"→"fallecido") | `.str.strip().str.lower()` |
| 3.4 | Convertir fechas de texto a datetime | `pd.to_datetime()` |
| 3.5 | Convertir edades en meses/días a años | `.loc[]` con máscara + operaciones |
| 3.6 | Verificar duplicados | `.duplicated()`, `.drop_duplicates()` |
| 3.7 | Imputar nulos categóricos | `.fillna()` |
| 3.8 | Mapear códigos de etnia a nombres | `.map()` |

---

## 4. Construcción de variables derivadas

Los datos crudos no siempre tienen las variables que necesitamos para un análisis.  
Una parte fundamental del trabajo con datos es **crear nuevas variables** a partir de las existentes.

### 4.1 Grupo de edad

Agrupar la edad en categorías facilita el análisis. Usamos `pd.cut()` para crear intervalos.

In [23]:
df_clean["grupo_edad"] = pd.cut(
    df_clean["edad_anios"],
    bins=[0, 18, 35, 50, 65, 120],
    labels=["0-18", "19-35", "36-50", "51-65", "66+"],
    include_lowest=True
)

print("Distribución por grupo de edad:")
print(df_clean["grupo_edad"].value_counts().sort_index())

Distribución por grupo de edad:
grupo_edad
0-18     10169
19-35    47163
36-50    32519
51-65    21813
66+      12118
Name: count, dtype: int64


### 4.2 Variable indicadora: fallecido

Crear una variable binaria (0/1) es muy útil para calcular tasas.  
Si `recuperado == "fallecido"`, entonces `fallecido = 1`.

In [24]:
df_clean["fallecido"] = (df_clean["recuperado"] == "fallecido").astype(int)

print("Distribución de fallecidos:")
print(df_clean["fallecido"].value_counts())
print(f"\nTasa de mortalidad general: {df_clean['fallecido'].mean() * 100:.2f}%")

Distribución de fallecidos:
fallecido
0    120287
1      3495
Name: count, dtype: int64

Tasa de mortalidad general: 2.82%


### 4.3 Días entre síntomas y diagnóstico

Al restar dos columnas de tipo `datetime`, pandas devuelve un `timedelta`.  
Con `.dt.days` lo convertimos a número entero de días.

In [25]:
df_clean["dias_sintomas_diagnostico"] = (
    df_clean["fecha_de_diagnostico"] - df_clean["fecha_de_inicio_de_sintomas"]
).dt.days

print("Estadísticas de días entre síntomas y diagnóstico:")
print(df_clean["dias_sintomas_diagnostico"].describe())

Estadísticas de días entre síntomas y diagnóstico:
count    114461.000000
mean          9.586995
std           5.958521
min           0.000000
25%           4.000000
50%           9.000000
75%          14.000000
max         145.000000
Name: dias_sintomas_diagnostico, dtype: float64


### 4.4 Días de recuperación

¿Cuántos días tardaron los pacientes en recuperarse desde el diagnóstico?

In [26]:
df_clean["dias_recuperacion"] = (
    df_clean["fecha_de_recuperacion"] - df_clean["fecha_de_diagnostico"]
).dt.days

print("Estadísticas de días de recuperación (desde diagnóstico):")
print(df_clean["dias_recuperacion"].describe())

Estadísticas de días de recuperación (desde diagnóstico):
count    119605.000000
mean         21.371673
std          37.971722
min           0.000000
25%           8.000000
50%          13.000000
75%          18.000000
max         382.000000
Name: dias_recuperacion, dtype: float64


### 4.5 Días hasta la muerte (solo fallecidos)

Para los casos que fallecieron, ¿cuántos días pasaron desde el inicio de síntomas hasta la muerte?

In [27]:
df_clean["dias_hasta_muerte"] = (
    df_clean["fecha_de_muerte"] - df_clean["fecha_de_inicio_de_sintomas"]
).dt.days

# Solo tiene sentido para fallecidos
print("Días desde síntomas hasta muerte (solo fallecidos):")
print(df_clean.loc[df_clean["fallecido"] == 1, "dias_hasta_muerte"].describe())

Días desde síntomas hasta muerte (solo fallecidos):
count    3495.000000
mean       20.211159
std        26.486068
min         0.000000
25%         9.000000
50%        16.000000
75%        24.000000
max       372.000000
Name: dias_hasta_muerte, dtype: float64


### 4.6 Extraer mes y año del reporte

Con `.dt.month` y `.dt.year` podemos extraer componentes de una fecha.  
Esto es muy útil para analizar tendencias temporales.

In [28]:
df_clean["mes_reporte"] = df_clean["fecha_reporte_web"].dt.month
df_clean["anio_reporte"] = df_clean["fecha_reporte_web"].dt.year

# Crear etiqueta año-mes para facilitar análisis temporal
df_clean["anio_mes"] = df_clean["fecha_reporte_web"].dt.to_period("M")

print("Casos por año:")
print(df_clean["anio_reporte"].value_counts().sort_index())

print("\nCasos por mes (todos los años):")
print(df_clean["mes_reporte"].value_counts().sort_index())

Casos por año:
anio_reporte
2020    110557
2021     13206
2022        19
Name: count, dtype: int64

Casos por mes (todos los años):
mes_reporte
1     12970
2         5
3         5
4         1
5      3777
6      3410
7     14436
8     22476
9     14082
10    15686
11    16103
12    20831
Name: count, dtype: int64


### Resumen de variables creadas

| Variable nueva | Tipo | Descripción |
|---|---|---|
| `edad_anios` | Numérica | Edad convertida a años (corrige meses y días) |
| `etnia` | Categórica | Nombre descriptivo del grupo étnico |
| `grupo_edad` | Categórica | Rangos de edad (0-18, 19-35, 36-50, 51-65, 66+) |
| `fallecido` | Binaria (0/1) | 1 si el paciente falleció |
| `dias_sintomas_diagnostico` | Numérica | Días entre inicio de síntomas y diagnóstico |
| `dias_recuperacion` | Numérica | Días entre diagnóstico y recuperación |
| `dias_hasta_muerte` | Numérica | Días entre síntomas y muerte (solo fallecidos) |
| `mes_reporte` | Numérica | Mes del reporte |
| `anio_reporte` | Numérica | Año del reporte |
| `anio_mes` | Period | Año-mes del reporte |

In [29]:
# Verificar las dimensiones y columnas finales
print("Dimensiones finales:", df_clean.shape)
print("\nColumnas del dataset limpio:")
for i, col in enumerate(df_clean.columns):
    print(f"  {i:2d}. {col}")

Dimensiones finales: (123782, 31)

Columnas del dataset limpio:
   0. fecha_reporte_web
   1. id_de_caso
   2. fecha_de_notificacion
   3. codigo_divipola_departamento
   4. nombre_departamento
   5. codigo_divipola_municipio
   6. nombre_municipio
   7. edad
   8. unidad_de_medida_de_edad
   9. sexo
  10. tipo_de_contagio
  11. ubicacion_del_caso
  12. estado
  13. recuperado
  14. fecha_de_inicio_de_sintomas
  15. fecha_de_muerte
  16. fecha_de_diagnostico
  17. fecha_de_recuperacion
  18. tipo_de_recuperacion
  19. pertenencia_etnica
  20. nombre_del_grupo_etnico
  21. edad_anios
  22. etnia
  23. grupo_edad
  24. fallecido
  25. dias_sintomas_diagnostico
  26. dias_recuperacion
  27. dias_hasta_muerte
  28. mes_reporte
  29. anio_reporte
  30. anio_mes


---
## 5. Análisis exploratorio con indexación

Ahora que los datos están limpios y tenemos variables nuevas, podemos hacer preguntas interesantes.

### Recordatorio de herramientas

| Herramienta | ¿Qué hace? | Ejemplo |
|---|---|---|
| `df[condicion]` | Filtra filas | `df[df["edad_anios"] > 60]` |
| `df.loc[filas, columnas]` | Selecciona por etiqueta | `df.loc[:5, ["edad_anios", "sexo"]]` |
| `df.iloc[filas, columnas]` | Selecciona por posición | `df.iloc[:5, :3]` |
| `&` (and), `|` (or) | Combina condiciones | `df[(cond1) & (cond2)]` |
| `.value_counts()` | Conteo de frecuencias | `df["sexo"].value_counts()` |
| `.mean()`, `.sum()`, `.median()` | Estadísticas sobre una columna filtrada | `df[df["sexo"]=="f"]["edad_anios"].mean()` |
| `len(df_filtrado)` | Contar filas de un filtro | `len(df[df["edad_anios"] > 60])` |

### Ejemplo 1: Filtrar casos en Bogotá y calcular su proporción

In [30]:
# Filtrar solo los casos de Bogotá
bogota = df_clean[df_clean["nombre_departamento"] == "bogota"]

print(f"Casos en Bogotá: {len(bogota):,}")
print(f"Total de casos: {len(df_clean):,}")
print(f"Porcentaje: {len(bogota) / len(df_clean) * 100:.2f}%")

Casos en Bogotá: 34,094
Total de casos: 123,782
Porcentaje: 27.54%


### Ejemplo 2: Top 10 departamentos con más casos

In [31]:
# value_counts() cuenta cuántas veces aparece cada valor
top_deptos = df_clean["nombre_departamento"].value_counts().head(10)
print("Top 10 departamentos con más casos:")
top_deptos

Top 10 departamentos con más casos:


nombre_departamento
bogota             34094
antioquia          18070
valle              10123
cundinamarca        5382
santander           4970
tolima              4044
barranquilla        4043
cartagena           3970
norte santander     3280
meta                3090
Name: count, dtype: int64

### Ejemplo 3: Tasa de mortalidad en mayores de 65 años vs. el total

Para calcular tasas podemos: filtrar un subconjunto y calcular `.mean()` sobre la variable indicadora (0/1).

In [32]:
# Tasa general
tasa_general = df_clean["fallecido"].mean() * 100

# Filtrar mayores de 65 y calcular su tasa
mayores_65 = df_clean[df_clean["edad_anios"] > 65]
tasa_mayores = mayores_65["fallecido"].mean() * 100

print(f"Tasa de mortalidad general:      {tasa_general:.2f}%")
print(f"Tasa de mortalidad en >65 años:  {tasa_mayores:.2f}%")
print(f"Fallecidos >65 años: {mayores_65['fallecido'].sum()} de {len(mayores_65)} casos")

Tasa de mortalidad general:      2.82%
Tasa de mortalidad en >65 años:  18.58%
Fallecidos >65 años: 2252 de 12118 casos


### Ejemplo 4: Seleccionar filas y columnas específicas con `.loc[]` e `.iloc[]`

In [33]:
# .loc[] selecciona por ETIQUETAS (nombres de columnas, índices)
# Primeras 5 filas, solo 3 columnas específicas
print("Con .loc[] — filas 0 a 4, columnas específicas:")
print(df_clean.loc[:4, ["nombre_departamento", "edad_anios", "sexo"]])

print("\n" + "=" * 50)

# .iloc[] selecciona por POSICIÓN (números enteros)
# Filas 10 a 14, columnas en posiciones 4, 7, 9
print("Con .iloc[] — filas 10 a 14, columnas por posición:")
print(df_clean.iloc[10:15, [4, 7, 9]])

Con .loc[] — filas 0 a 4, columnas específicas:
  nombre_departamento  edad_anios sexo
0               valle        67.0    f
1               valle        66.0    f
2               valle        68.0    f
3               valle        74.0    f
4               valle        65.0    f

Con .iloc[] — filas 10 a 14, columnas por posición:
   nombre_departamento  edad sexo
10               valle    62    f
11               valle    36    f
12               valle    36    f
13               valle    35    f
14           antioquia    37    f


---
## 6. Actividad en clase — Preguntas de indexación y análisis

Usa el DataFrame `df_clean` para responder las siguientes preguntas.  
Cada una requiere **indexación, filtrado, `value_counts()` o cálculos sobre columnas filtradas**.

---

### Nivel 1: Filtrado básico

**Pregunta 1:** ¿Cuántas mujeres (`sexo == "f"`) hay en el dataset? ¿Y cuántos hombres (`sexo == "m"`)? ¿Cuál es la proporción de cada uno?



In [ ]:
# ESCRIBE AQUÍ:


**Pregunta 2:** ¿Cuántos casos fueron de contagio importado (`tipo_de_contagio == "importado"`)? Muestra con `.value_counts()` de qué departamentos venían esos casos.

In [ ]:
# ESCRIBE AQUÍ:


**Pregunta 3:** Filtra los casos del departamento de Antioquia (`nombre_departamento == "antioquia"`). ¿Cuántos hay? ¿Cuántos de ellos fallecieron? ¿Cuál es la tasa de mortalidad?



In [ ]:
# ESCRIBE AQUÍ:


**Pregunta 4:** Usa `.loc[]` para mostrar las filas 100 a 105, pero **solo** las columnas `nombre_departamento`, `nombre_municipio`, `edad_anios`, `sexo` y `recuperado`.

In [ ]:
# ESCRIBE AQUÍ:


---
### Nivel 2: Filtros combinados y cálculos

**Pregunta 5:** Filtra los casos de menores de 18 años que fallecieron (dos condiciones con `&`). ¿Cuántos hay? Muestra las columnas `edad_anios`, `sexo`, `nombre_departamento` y `fecha_de_muerte`.



In [ ]:
# ESCRIBE AQUÍ:


**Pregunta 6:** ¿Cuál es la edad promedio y la mediana de los fallecidos? ¿Y de los recuperados? Compara ambos resultados.



In [ ]:
# ESCRIBE AQUÍ:


**Pregunta 7:** Filtra los casos de Bogotá que fueron de contagio comunitario (`tipo_de_contagio == "comunitaria"`). ¿Cuántos son? ¿Qué porcentaje de los casos de Bogotá representan?

*Pista: necesitas dos condiciones con `&`. Para el porcentaje divide por el total de Bogotá.*

In [ ]:
# ESCRIBE AQUÍ:


**Pregunta 8:** ¿Cuál es el promedio de días de recuperación (`dias_recuperacion`) de los hombres recuperados? ¿Y de las mujeres recuperadas? ¿Quién se recupera más rápido?

*Pista: filtra por `recuperado == "recuperado"` AND `sexo == "m"` (o `"f"`), luego filtra `dias_recuperacion > 0` y calcula `.mean()`.*

In [ ]:
# ESCRIBE AQUÍ:


---
### Nivel 3: Análisis más profundo

**Pregunta 9:** Compara la tasa de mortalidad entre Bogotá y el Valle del Cauca. Para cada uno:
- Filtra el departamento
- Calcula `fallecido.sum()` y `fallecido.mean() * 100`

¿En cuál departamento la mortalidad es más alta?

In [ ]:
# ESCRIBE AQUÍ:


**Pregunta 10:** ¿Cuántos casos se reportaron en el año 2021? ¿Y cuántos fallecidos hubo en 2021? Calcula la tasa de mortalidad solo para ese año y compárala con la de 2020.



In [ ]:
# ESCRIBE AQUÍ:


**Pregunta 11:** Filtra los hombres mayores de 65 años que fallecieron en Bogotá. ¿Cuántos son? ¿Cuál fue el promedio de días hasta la muerte (`dias_hasta_muerte`) para ese grupo?

*Pista: necesitas combinar 4 condiciones con `&`.*

In [ ]:
# ESCRIBE AQUÍ:


---
### Nivel 4: Desafío — Construye tu propia variable y analiza

**Pregunta 12:** Crea una variable `diagnostico_rapido` que sea `1` si el diagnóstico ocurrió en 3 días o menos desde el inicio de síntomas, y `0` en caso contrario.

Luego responde: ¿Cuál es la tasa de mortalidad de los que tuvieron diagnóstico rápido? ¿Y la de los que no? ¿El diagnóstico rápido se asocia con menor mortalidad?



In [ ]:
# ESCRIBE AQUÍ:
